In [1]:
from datetime import datetime

print(f"Timestamp: {datetime.now()}")

Timestamp: 2025-11-13 14:58:11.163236


# Nuclear and whole cell 

* https://www.10xgenomics.com/support/software/xenium-onboard-analysis/3.1/tutorials/outputs/xoa-output-understanding-outputs
* https://www.10xgenomics.com/support/software/xenium-onboard-analysis/3.1/advanced/example-code
  

Go to "Transcript data"

"The transcripts file (transcripts.parquet) is provided in Parquet format to enable faster loading and reading of data (code examples here). It contains data to evaluate transcript quality and localization. The file contains one row for each decoded transcript"


Basically we need to subset on nuclear transcripts which are stored in the parquet file. So after creating the anndata split it into nuclear and whole-cell branches.

In [2]:
from pathlib import Path
import dask.dataframe as dd
import os
import numpy as np
import pandas as pd
import anndata as ad
import matplotlib.pyplot as plt

In [3]:
base_dir = Path("/home/workspace/projects/drg")

h5ad_dir     = base_dir / "data/h5ad/export_01/01_raw/output_folder"
whole_dir = h5ad_dir
xenium_root  = base_dir / "download/isolon"

out_main     = base_dir / "data/h5ad/export_01/01_raw/output_folder/nuclear_added"
out_nuclear  = base_dir / "data/h5ad/export_01/01_raw/output_folder/nuclear_only"
nuc_dir = out_nuclear


out_main.mkdir(parents=True, exist_ok=True)
out_nuclear.mkdir(parents=True, exist_ok=True)

In [4]:
#whole_dir = base_dir / "data/h5ad/export_01/01_raw/output_folder"


In [5]:
def load_nuclear_matrix(parquet_path, genes, cells):
    df = dd.read_parquet(parquet_path)
    nuc = df[df["overlaps_nucleus"] == 1].compute()
    mat = (
        nuc.groupby(["cell_id", "feature_name"])
        .size()
        .unstack(fill_value=0)
    )
    mat = mat.reindex(index=cells).fillna(0)
    mat = mat.reindex(columns=genes).fillna(0)
    return mat

for fname in os.listdir(h5ad_dir):

    if not fname.endswith(".h5ad"):
        continue

    print(f"\nProcessing {fname} ...")

    # Load whole-cell h5ad
    adata = ad.read_h5ad(f"{h5ad_dir}/{fname}")

    # Find matching transcripts.parquet
    run_id = adata.obs["xenium_run_id"].iloc[0]
    parquet_path = f"{xenium_root}/{run_id}/transcripts.parquet"

    genes = adata.var_names
    cells = adata.obs_names

    # Compute nuclear counts
    nuc_df = load_nuclear_matrix(parquet_path, genes, cells)
    nuclear = nuc_df.to_numpy()

    # Add nuclear layer to whole-cell data
    adata.layers["nuclear"] = nuclear

    # Compute nuclear fraction
    X = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
    adata.obs["nuclear_fraction"] = nuclear.sum(1) / (X.sum(1) + 1e-9)

    # Save whole-cell + nuclear-layer version
    adata.write(f"{out_main}/{fname}")

    # ---- Create nuclear-only AnnData ----
    adata_nuc = ad.AnnData(
        X=nuclear,
        obs=adata.obs.copy(),
        var=adata.var.copy(),
        obsm=adata.obsm.copy(),
        uns=adata.uns.copy(),
    )
    adata_nuc.write(f"{out_nuclear}/{fname.replace('.h5ad','_nuclear.h5ad')}")

    print(f"Saved full-cell + nuclear layer: {out_main}/{fname}")
    print(f"Saved nuclear-only:              {out_nuclear}/{fname.replace('.h5ad','_nuclear.h5ad')}")


Processing TMA00304.h5ad ...
Saved full-cell + nuclear layer: /home/workspace/projects/drg/data/h5ad/export_01/01_raw/output_folder/nuclear_added/TMA00304.h5ad
Saved nuclear-only:              /home/workspace/projects/drg/data/h5ad/export_01/01_raw/output_folder/nuclear_only/TMA00304_nuclear.h5ad

Processing TMA00306.h5ad ...
Saved full-cell + nuclear layer: /home/workspace/projects/drg/data/h5ad/export_01/01_raw/output_folder/nuclear_added/TMA00306.h5ad
Saved nuclear-only:              /home/workspace/projects/drg/data/h5ad/export_01/01_raw/output_folder/nuclear_only/TMA00306_nuclear.h5ad

Processing TMA00303.h5ad ...
Saved full-cell + nuclear layer: /home/workspace/projects/drg/data/h5ad/export_01/01_raw/output_folder/nuclear_added/TMA00303.h5ad
Saved nuclear-only:              /home/workspace/projects/drg/data/h5ad/export_01/01_raw/output_folder/nuclear_only/TMA00303_nuclear.h5ad

Processing TMA00305.h5ad ...
Saved full-cell + nuclear layer: /home/workspace/projects/drg/data/h5ad/e